In [1]:
"""
Objective: Improve on an existing prompt by using the prompt engineering topics discussed in the Prompt Engineering Techniques section
"""

# Identify changes in source files and reload
%load_ext autoreload
%autoreload 2

# Import API key and Anthropic API
from dotenv import load_dotenv
from anthropic import Anthropic

import json

"""
Import Helper functions for:
- Chat
- Dataset generation
"""
from claude_chat import ClaudeChat
from claude_dataset import ClaudeDataset
from claude_evaluation import ClaudeEvaluation

In [2]:
# Access the API key
load_dotenv()

# Access the Anthropic API
client = Anthropic()

# Specify the model Claude will use
model = "claude-sonnet-5"

# Conversation history for analyzing scholarly articles for the main topics
article_topics_conversation = []

"""
Access:
- Conversation history related to scholarly article topics
- Functions to store user inputs
- Functions to store Claude responses
"""
claude_chat = ClaudeChat(model, client, article_topics_conversation)

"""
Access:
- Conversation history related to generating dataset JSON data
- Functions to store dataset JSON data in a JSON file
- Functiosn to load dataset JSON data from a JSON file
"""
generated_dataset = ClaudeDataset(model, client)

claude_evaluation = ClaudeEvaluation(client, model)

In [3]:
"""
Prompt engineering rules to replace prefilling
Rules for generating the JSON list of scholarly article topics
"""
scholarly_topics_prompt_rules = """
Rules:
- Return only valid JSON
- Do not use markdown
- Do not include comments
- Do not include explanations
- After the plain text, write END_OF_COMMANDS
"""

dataset_generation_prompt_rules = """
Rules:
- Return only valid JSON
- Do not use markdown
- Do not include comments
- Do not include explanations outside the JSON
- Do not include trailing commas
- After the JSON, write END_OF_COMMANDS
"""

# Provide Claude with context to customize how Claude responds to user inputs
evaluation_system_prompt = """
You are an expert at analyzing passages of scholarly text and extracting all the main topics from the text
"""

claude_chat.stop_sequences.append("END_OF_COMMANDS")

In [4]:
# Load in sample scholarly text
f = open("short_scholarly_text.txt")
scholarly_text = f.read()

In [5]:
# Prompting Process: 1. Create a draft of the prompt
prompt_draft = f"""
What topics are in here?

{scholarly_text}

{scholarly_topics_prompt_rules}
"""

claude_chat.userInput(prompt_draft)

claude_response = claude_chat.askClaude(evaluation_system_prompt)

# Parse just to see the output better
parsed_response = json.loads(claude_response)

# Observe the output from the initial prompt draft
print(json.dumps(parsed_response, indent=2))

{
  "topics": [
    "Liberal arts education and its true purpose",
    "Consciousness and awareness in daily life",
    "Self-centeredness as a default human setting",
    "The nature of perception and constructed meaning",
    "Freedom of choice in thought and interpretation",
    "Critical thinking versus capacity to think",
    "Religious belief versus atheism and interpretation of experience",
    "Arrogance and intellectual humility",
    "Boredom, routine, and frustration in adult life",
    "Worship as a universal human behavior",
    "Choosing what to worship (money, power, body, intellect)",
    "Empathy and compassion toward others",
    "Mental discipline and control over thought",
    "Suicide and psychological unconsciousness",
    "Consumerism and everyday adult drudgery",
    "Well-adjustedness and psychological adaptation",
    "The concept of 'default setting' in human cognition",
    "True freedom versus superficial freedom",
    "Attention as a form of discipline",
 

In [6]:
# Prompting Process: 2. Generate test data
dataset_prompt = f"""
Generate a small evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that extract all the topics into a JSON array of strings. The topics focus on what the article talks about. Generate an array of JSON objects, each representing a paragraph from a scholarly journal written in English.

Example output:
[
	{{
		"content": "A very short scholarly-style paragraph between 30 and 60 words.",
        "format": "text",
        "solution_criteria": "A very short key criteria for evaluating the solution between 20 and 40 words."
	}},
    ...
]
 
 Please generate 3 objects
 
 {dataset_generation_prompt_rules}
"""

dataset_system_prompt = """
You are an expert in creating accurate, useful, and exhaustive evaluation datasets.
You follow industry best practices when it comes to generating prompt evaluation datasets
"""

# Add the dataset prompt to the message history
generated_dataset.storeUserInput(dataset_prompt)

generated_dataset.dataset_file = "dataset.json"

generated_dataset.stop_sequences.append("END_OF_COMMANDS")

# Generate the dataset and store it in dataset.json
generated_dataset.generateDataset(dataset_system_prompt)

# Observe the output
dataset = json.dumps(generated_dataset.getEvaluationDataset(), indent=2)

print(dataset)

[
  {
    "content": "The rapid proliferation of microplastics in marine ecosystems has become a critical environmental concern. Recent studies indicate that these particles accumulate in the tissues of filter-feeding organisms, potentially disrupting endocrine functions and bioaccumulating through trophic levels, ultimately posing risks to both marine biodiversity and human health via seafood consumption.",
    "format": "text",
    "solution_criteria": "Should include topics such as microplastics, marine ecosystems, bioaccumulation, endocrine disruption, and human health risks from seafood consumption."
  },
  {
    "content": "Recent advances in CRISPR-Cas9 gene editing have enabled precise modifications of the human genome, offering promising therapeutic avenues for genetic disorders such as sickle cell anemia. However, ethical concerns regarding germline editing and off-target mutations continue to challenge widespread clinical adoption, necessitating robust regulatory frameworks.

In [11]:
# Prompting Process: 3. Evaluate the prompt

# Use the Claude generated dataset
dataset = generated_dataset.getEvaluationDataset()

# Run the dataset against the initial prompt draft
dataset_prompt_results = claude_evaluation.testPrompt(dataset, dataset_generation_prompt_rules)

# Use model based grading to evaluate Claude's responses to the initial prompt draft
model_grading_results = claude_evaluation.modelBasedGrading(dataset, dataset_prompt_results, dataset_generation_prompt_rules)

# Observe the grading values
print("Model Based Grading Results")
print(json.dumps(model_grading_results, indent=2))

# Use code based grading to verify the generated code has valid syntax and follows the correct format
code_grading_results = claude_evaluation.codeBasedGrading(dataset_prompt_results)

# Observe the grading values
print("Code Based Grading Results")
print(json.dumps(code_grading_results, indent=2))

# Calcuate the mean of the model based grading and the code based grading
mean = claude_evaluation.calculateAverage(model_grading_results, code_grading_results)

# Observe the average
print(f"Prompt Evaluation Average: {mean})

Model Based Grading Results
[
  "{\"strengths\": [\"Covers all required topics\", \"Includes relevant additional context like trophic levels\"], \"weaknesses\": [\"Simple list format lacks depth or explanation\"], \"reasoning\": \"The solution comprehensively addresses all criteria topics with relevant related concepts.\", \"score\": 9}",
  "{\"strengths\": [\"Covers all required topics\", \"Includes relevant extra details like off-target mutations\"], \"weaknesses\": [\"Slightly redundant with clinical adoption overlapping ethical concerns\"], \"reasoning\": \"The solution comprehensively addresses all specified criteria with appropriate topic granularity.\", \"score\": 9}",
  "{\"strengths\": [\"Covers all required topics\", \"Adds relevant related concepts\"], \"weaknesses\": [\"Slight redundancy with generic terms\"], \"reasoning\": \"The solution comprehensively covers all required topics with relevant additions.\", \"score\": 9}"
]
Code Based Grading Results
[
  {
    "solution":